# Fase 2e: Decision Engine

**Integra las salidas de todas las fases anteriores en una recomendación accionable por cliente.**

### Lógica de decisión (5 pasos + conflict resolution)
1. **Prioridad:** basada en uplift_x (CATE) directo — no P(canje) × uplift
2. **Objetivo según funnel:** estado actual → acción correspondiente
3. **Personalizar por cluster:** segmento conductual → tipo de incentivo
4. **Retailer + canal:** P(retailer) + canal preferido
5. **Timing:** urgencia según puntos por vencer, estancamiento, probabilidad
6. **Conflict resolution:** funnel overrides cluster; uplift negativo → no contactar

### Output por cliente
Prioridad, objetivo, acción, incentivo, retailer, canal, timing, expected_value, justificación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors
from scipy.stats import skew

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("Imports OK")

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────
USE_MOCK = True

if USE_MOCK:
    import duckdb
    with open("../../fase1/test_mock_local.py") as f:
        code = f.read().replace("con.close()", "# con.close()")
    _globals = {}
    exec(code, _globals)
    con = _globals['con']
    df = con.execute("SELECT * FROM customer_snapshot").df()
    
    # Spending post para uplift
    spending_post = con.execute("""
        WITH snap AS (
            SELECT DISTINCT cust_id, t0, t0 + INTERVAL '6 months' AS t0_end
            FROM customer_snapshot
        )
        SELECT s.cust_id, s.t0,
               COALESCE(SUM(t.tran_amt), 0) AS spending_post_6m
        FROM snap s
        LEFT JOIN mock_transaction_entity t
            ON t.cust_id = s.cust_id
            AND t.tran_date >= s.t0
            AND t.tran_date < s.t0_end
        GROUP BY s.cust_id, s.t0
    """).df()
    con.close()
else:
    from google.cloud import bigquery
    client = bigquery.Client(project="my-gcp-project")
    df = client.query("SELECT * FROM `my-gcp-project.loyalty_analytics.customer_snapshot`").to_dataframe()
    spending_post = pd.DataFrame()

df['t0'] = pd.to_datetime(df['t0'])
spending_post['t0'] = pd.to_datetime(spending_post['t0'])
df = df.merge(spending_post, on=['cust_id', 't0'], how='left')
df['spending_post_6m'] = df['spending_post_6m'].fillna(0)
df['treatment'] = (df['y'] >= 1).astype(int)

print(f"Datos: {df.shape}, t0s: {sorted(df['t0'].dt.strftime('%Y-%m').unique())}")

## 0. Reconstruir componentes (Clustering + Uplift)

En producción estos vienen pre-calculados. Aquí los reconstruimos inline.

In [ ]:
# ── Reconstruir clustering (fase 2c) ─────────────────────────────────────
from sklearn.metrics import silhouette_score

CLUSTER_FEATURES = [
    'frequency_monthly_avg', 'monetary_monthly_avg', 'redeem_rate',
    'retailer_entropy', 'pct_redeem_digital', 'earn_velocity_90',
    'days_since_last_activity', 'points_pressure',
]

# Retailer entropy (frequency-based, matching 2c)
if 'retailer_entropy' not in df.columns:
    freqs = df[['freq_store_a', 'freq_store_b', 'freq_store_c', 'freq_store_d', 'freq_store_e']].values
    tot = np.where(freqs.sum(axis=1, keepdims=True) == 0, 1, freqs.sum(axis=1, keepdims=True))
    p = freqs / tot
    df['retailer_entropy'] = -(p * np.where(p > 0, np.log(p), 0)).sum(axis=1)

CLUSTER_FEATURES = [f for f in CLUSTER_FEATURES if f in df.columns]

# Contextual fillna matching fase2c
df_clust = df[CLUSTER_FEATURES].copy()
df_clust['days_since_last_activity'] = df_clust['days_since_last_activity'].fillna(999)
df_clust['pct_redeem_digital'] = df_clust['pct_redeem_digital'].fillna(0)
df_clust['redeem_rate'] = df_clust['redeem_rate'].fillna(0)
df_clust[CLUSTER_FEATURES] = df_clust[CLUSTER_FEATURES].fillna(0)

# Log-transform + winsorize + scale (same as 2c)
t0_train = df['t0'].min()
mask_train = df['t0'] == t0_train

skews = df_clust.loc[mask_train].apply(skew)
log_feats = skews[skews.abs() > 2].index.tolist()
for feat in log_feats:
    df_clust[feat] = np.log1p(df_clust[feat])

for feat in CLUSTER_FEATURES:
    lo = df_clust.loc[mask_train, feat].quantile(0.01)
    hi = df_clust.loc[mask_train, feat].quantile(0.99)
    df_clust[feat] = df_clust[feat].clip(lo, hi)

cl_scaler = StandardScaler()
cl_scaler.fit(df_clust.loc[mask_train])
X_cl = cl_scaler.transform(df_clust)

# Dynamic K selection (same as 2c)
K_RANGE = range(2, 9)
silhouettes = []
for k in K_RANGE:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_tmp = km_tmp.fit_predict(X_cl[mask_train.values])
    silhouettes.append(silhouette_score(X_cl[mask_train.values], labels_tmp))

# Prefer K in [4,6] per business spec
preferred = {k: s for k, s in zip(K_RANGE, silhouettes) if 4 <= k <= 6}
K_OPT = max(preferred, key=preferred.get) if preferred else int(K_RANGE[np.argmax(silhouettes)])
print(f"K_OPT={K_OPT} (silhouette={preferred.get(K_OPT, max(silhouettes)):.4f})")

kmeans = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
kmeans.fit(X_cl[mask_train.values])
df['cluster'] = kmeans.predict(X_cl)

# Auto-name (simplified — use profiles on train only)
from scipy.optimize import linear_sum_assignment
profiles = df.loc[mask_train].groupby('cluster')[CLUSTER_FEATURES].mean()
ranges = profiles.max() - profiles.min()
pn = (profiles - profiles.min()) / ranges.where(ranges > 0, 1)  # avoid div-by-zero → 0/1=0
pn = pn.fillna(0)

archetypes = {
    'Dormidos': {'days_since_last_activity': 2.0, 'frequency_monthly_avg': -1.0, 'monetary_monthly_avg': -1.0},
    'Heavy Users': {'frequency_monthly_avg': 2.0, 'monetary_monthly_avg': 1.5, 'retailer_entropy': 1.0},
    'Cazadores de Canje': {'redeem_rate': 2.5, 'points_pressure': 1.0},
    'Exploradores': {'retailer_entropy': 1.5, 'frequency_monthly_avg': 0.5, 'redeem_rate': -1.0},
    'Digitales': {'pct_redeem_digital': 2.5, 'redeem_rate': 0.5},
    'En Riesgo': {'days_since_last_activity': 1.5, 'points_pressure': 1.5, 'redeem_rate': 0.5},
}

cluster_ids = list(pn.index)
arch_names = list(archetypes.keys())
score_m = np.zeros((len(cluster_ids), len(arch_names)))
for i, c in enumerate(cluster_ids):
    for j, name in enumerate(arch_names):
        for feat, w in archetypes[name].items():
            if feat in pn.columns:
                score_m[i, j] += w * pn.loc[c, feat]

row_idx, col_idx = linear_sum_assignment(-score_m)
CLUSTER_NAMES = {cluster_ids[r]: arch_names[c] for r, c in zip(row_idx, col_idx)}
df['cluster_name'] = df['cluster'].map(CLUSTER_NAMES)

print("Clustering reconstruido:")
for c, name in sorted(CLUSTER_NAMES.items()):
    print(f"  {c} = {name} ({(df['cluster']==c).sum():,})")

In [ ]:
# ── Reconstruir uplift (fase 2d) ─────────────────────────────────────────
from sklearn.metrics import roc_auc_score

PSM_FEATURES = [
    'frequency_monthly_avg', 'monetary_monthly_avg', 'redeem_rate',
    'retailer_entropy', 'pct_redeem_digital', 'earn_velocity_90',
    'days_since_last_activity', 'points_pressure',
    'stock_points_at_t0', 'redeem_count_pre', 'frequency_total', 'monetary_total',
    'tenure_months',
]
PSM_FEATURES = [f for f in PSM_FEATURES if f in df.columns]

X_psm = df[PSM_FEATURES].fillna(0)
T = df['treatment'].values
Y = df['spending_post_6m'].values

# Propensity score
ps_scaler = StandardScaler()
X_ps = ps_scaler.fit_transform(X_psm)
ps_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
ps_model.fit(X_ps, T)
df['propensity_score'] = ps_model.predict_proba(X_ps)[:, 1]

# X-Learner CATE
model_t1 = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)
model_t0 = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)
model_t1.fit(X_ps[T==1], Y[T==1])
model_t0.fit(X_ps[T==0], Y[T==0])

mu1 = model_t1.predict(X_ps)
mu0 = model_t0.predict(X_ps)

D1 = Y[T==1] - model_t0.predict(X_ps[T==1])
D0 = model_t1.predict(X_ps[T==0]) - Y[T==0]

tau1 = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
tau0 = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
tau1.fit(X_ps[T==1], D1)
tau0.fit(X_ps[T==0], D0)

e = df['propensity_score'].values
df['uplift_x'] = e * tau0.predict(X_ps) + (1 - e) * tau1.predict(X_ps)

# EV informational only — NOT used for prioritization (see Paso 1)
df['expected_value'] = df['propensity_score'] * df['uplift_x']

# Lift GitLab por quintil
if 'quintil_gasto' in df.columns:
    Q_MAP = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4', 5: 'Q5'}
    df['quintil_label'] = df['quintil_gasto'].map(Q_MAP)
else:
    df['quintil_label'] = pd.qcut(df['monetary_total'], 5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')

# GitLab lift per quintil
gl_lifts = {}
for q in df['quintil_label'].dropna().unique():
    mask_q = df['quintil_label'] == q
    canj = df.loc[mask_q & (df['treatment']==1), 'spending_post_6m']
    pot = df.loc[mask_q & (df['treatment']==0) & (df['stock_points_at_t0']>=1000), 'spending_post_6m']
    if len(canj) >= 5 and len(pot) >= 5:
        gl_lifts[q] = (canj.mean() - pot.mean()) / max(pot.mean(), 1)

df['lift_gitlab_quintil'] = df['quintil_label'].map(gl_lifts)

ps_auc = roc_auc_score(T, ps_model.predict_proba(X_ps)[:, 1])
print(f"Uplift reconstruido: propensity AUC={ps_auc:.3f}")
print(f"  uplift_x mean=${df['uplift_x'].mean():,.0f}, EV mean=${df['expected_value'].mean():,.0f}")
print(f"  lift_gitlab por quintil: {gl_lifts}")

## 1. Decision Engine — 5 Pasos

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASO 1: PRIORIDAD basada en uplift_x (CATE) directo
# ══════════════════════════════════════════════════════════════════════════
# Priorizar por uplift (CATE) directo, no por P(canje) × uplift
# P(canje) alta = cliente que canjea solo → bajo valor marginal de intervención
# Filtrar uplift negativo → 'No contactar'
df['prioridad'] = 'Media'
df.loc[df['uplift_x'] <= 0, 'prioridad'] = 'No contactar'

uplift_pos = df.loc[df['uplift_x'] > 0, 'uplift_x']
if len(uplift_pos) > 0:
    q60 = uplift_pos.quantile(0.60)
    q80 = uplift_pos.quantile(0.80)
    df.loc[(df['uplift_x'] > 0) & (df['uplift_x'] <= q60), 'prioridad'] = 'Baja'
    df.loc[(df['uplift_x'] > q60) & (df['uplift_x'] <= q80), 'prioridad'] = 'Media'
    df.loc[df['uplift_x'] > q80, 'prioridad'] = 'Alta'

print("PASO 1: PRIORIDAD (basada en uplift_x directo)")
for p in ['Alta', 'Media', 'Baja', 'No contactar']:
    mask = df['prioridad'] == p
    print(f"  {p}: {mask.sum():,} ({mask.mean():.0%}) | uplift_x=${df.loc[mask,'uplift_x'].mean():,.0f} | EV=${df.loc[mask,'expected_value'].mean():,.0f} | tasa canje={df.loc[mask,'treatment'].mean():.1%}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASO 2: OBJETIVO según funnel state
# ══════════════════════════════════════════════════════════════════════════
OBJETIVO_FUNNEL = {
    'INSCRITO':           'Activar primera compra',
    'PARTICIPANTE':       'Acelerar acumulación de puntos',
    'POSIBILIDAD_CANJE':  'Empujar primer canje',
    'CANJEADOR':          'Generar recurrencia de canje',
    'RECURRENTE':         'Retener y aumentar ticket',
    'FUGA':               'Reactivar urgente',
}

df['objetivo'] = df['funnel_state_at_t0'].map(OBJETIVO_FUNNEL).fillna('Sin clasificar')

print("PASO 2: OBJETIVO POR FUNNEL")
obj_dist = df.groupby(['funnel_state_at_t0', 'objetivo']).size().reset_index(name='n')
obj_dist['pct'] = obj_dist['n'] / len(df) * 100
print(obj_dist.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASO 3: ACCION + INCENTIVO por cluster
# ══════════════════════════════════════════════════════════════════════════
ACCION_CLUSTER = {
    'Dormidos':            {'accion': 'Oferta directa de reactivación', 'incentivo': 'Puntos bonus x2 por primera compra'},
    'Exploradores':        {'accion': 'Educar sobre beneficios del canje', 'incentivo': 'Tutorial + primer canje con descuento'},
    'Cazadores de Canje':  {'accion': 'Descuento personalizado', 'incentivo': 'Canje con 20% menos puntos'},
    'Heavy Users':         {'accion': 'Experiencia premium exclusiva', 'incentivo': 'Acceso anticipado / evento VIP'},
    'Digitales':           {'accion': 'Oferta exclusiva canal digital', 'incentivo': 'Canje flash 24h solo en app'},
    'En Riesgo':           {'accion': 'Retención preventiva', 'incentivo': 'Extensión vencimiento + bonus por actividad'},
}

df['accion'] = df['cluster_name'].map(lambda x: ACCION_CLUSTER.get(x, {}).get('accion', 'Comunicación general'))
df['incentivo'] = df['cluster_name'].map(lambda x: ACCION_CLUSTER.get(x, {}).get('incentivo', 'Oferta estándar'))

print("PASO 3: ACCION POR CLUSTER")
for name in sorted(ACCION_CLUSTER.keys()):
    n = (df['cluster_name'] == name).sum()
    print(f"  {name} ({n:,}): {ACCION_CLUSTER[name]['accion']}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASO 4: RETAILER + CANAL preferido
# ══════════════════════════════════════════════════════════════════════════
# Retailer: usar dominant_retailer (ya calculado en snapshot)
df['retailer_recomendado'] = df['dominant_retailer'].replace('NINGUNO', 'STOREA')

# Canal: digital vs presencial basado en pct_redeem_digital
df['canal'] = np.where(df['pct_redeem_digital'] > 0.5, 'Digital', 'Presencial')

# Para clientes sin historial de canje, usar canal por contactabilidad
no_redeem = df['redeem_count_pre'] == 0
df.loc[no_redeem & df['contact_push_flg'], 'canal'] = 'Push'
df.loc[no_redeem & ~df['contact_push_flg'] & df['contact_email_flg'], 'canal'] = 'Email'

print("PASO 4: RETAILER + CANAL")
print(f"\nRetailer recomendado:")
print(df['retailer_recomendado'].value_counts().to_string())
print(f"\nCanal:")
print(df['canal'].value_counts().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASO 5: TIMING
# ══════════════════════════════════════════════════════════════════════════
# Urgente: puntos por vencer (points_pressure > 0.5)
# Pronto: estancado (days_since_last_activity > 180)
# Inmediato: P(canje) alta (propensity > 0.5)
# Normal: resto

def assign_timing(row):
    if row['points_pressure'] > 0.5:
        return 'Urgente (puntos por vencer)'
    if row['funnel_state_at_t0'] == 'FUGA':
        return 'Urgente (fuga)'
    if row['days_since_last_activity'] > 180:
        return 'Pronto (estancado)'
    if row['propensity_score'] > 0.5:
        return 'Inmediato (alta probabilidad)'
    return 'Normal'

df['timing'] = df.apply(assign_timing, axis=1)

print("PASO 5: TIMING")
print(df['timing'].value_counts().to_string())

# ── CONFLICT RESOLUTION ──────────────────────────────────────────────────
# Funnel overrides cluster action when they conflict
mask_inscrito = df['funnel_state_at_t0'] == 'INSCRITO'
df.loc[mask_inscrito, 'accion'] = 'Activar primera compra'
df.loc[mask_inscrito, 'incentivo'] = 'Puntos bonus por primera compra'

mask_fuga = df['funnel_state_at_t0'] == 'FUGA'
df.loc[mask_fuga, 'accion'] = 'Reactivación urgente'
df.loc[mask_fuga, 'incentivo'] = 'Oferta directa de reactivación + bonus puntos'

# Negative uplift → do not contact
mask_neg = df['uplift_x'] <= 0
df.loc[mask_neg, 'accion'] = 'No contactar (uplift negativo)'
df.loc[mask_neg, 'incentivo'] = '-'

print(f"\nCONFLICT RESOLUTION:")
print(f"  INSCRITO overrides: {mask_inscrito.sum():,}")
print(f"  FUGA overrides: {mask_fuga.sum():,}")
print(f"  Uplift negativo → no contactar: {mask_neg.sum():,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# JUSTIFICACION: texto legible por humano
# ══════════════════════════════════════════════════════════════════════════
def build_justificacion(row):
    parts = []
    parts.append(f"Prioridad {row['prioridad']} (EV=${row['expected_value']:,.0f})")
    parts.append(f"Funnel: {row['funnel_state_at_t0']} → {row['objetivo']}")
    parts.append(f"Segmento: {row['cluster_name']}")
    parts.append(f"P(canje)={row['propensity_score']:.0%}")
    parts.append(f"Uplift=${row['uplift_x']:,.0f}")
    if not pd.isna(row.get('lift_gitlab_quintil')):
        parts.append(f"Lift GitLab={row['lift_gitlab_quintil']:.1%}")
    return ' | '.join(parts)

df['justificacion'] = df.apply(build_justificacion, axis=1)

print("Ejemplo de justificación:")
print(df['justificacion'].iloc[0])

## 2. Output Final por Cliente

In [ ]:
# ── Tabla de decisión final ───────────────────────────────────────────────
OUTPUT_COLS = [
    'cust_id', 't0', 'prioridad', 'expected_value',
    'objetivo', 'accion', 'incentivo',
    'retailer_recomendado', 'canal', 'timing',
    'propensity_score', 'uplift_x', 'lift_gitlab_quintil',
    'cluster_name', 'funnel_state_at_t0', 'quintil_label',
    'stock_points_at_t0', 'points_pressure', 'days_since_last_activity',
    'justificacion',
]

OUTPUT_COLS = [c for c in OUTPUT_COLS if c in df.columns]
df_output = df[OUTPUT_COLS].copy()

# Ordenar por prioridad (Alta primero) y uplift_x descendente
prio_order = {'Alta': 0, 'Media': 1, 'Baja': 2, 'No contactar': 3}
df_output['_prio_sort'] = df_output['prioridad'].map(prio_order)
df_output = df_output.sort_values(['_prio_sort', 'expected_value'], ascending=[True, False]).drop(columns='_prio_sort')

print(f"OUTPUT: {df_output.shape[0]:,} filas × {df_output.shape[1]} columnas")
print(f"\nTop 10 clientes (mayor prioridad):")
top10_cols = ['cust_id', 't0', 'prioridad', 'expected_value', 'objetivo', 'accion',
              'retailer_recomendado', 'canal', 'timing', 'cluster_name']
top10_cols = [c for c in top10_cols if c in df_output.columns]
print(df_output[top10_cols].head(10).to_string(index=False))

In [ ]:
# ── Snapshot más reciente (lo que se ejecutaría en producción) ────────────
latest_t0 = df_output['t0'].max()
df_latest = df_output[df_output['t0'] == latest_t0].copy()

print(f"LISTA DE ACCION — t0={latest_t0.strftime('%Y-%m-%d')} ({len(df_latest):,} clientes)")
print("="*80)

print(f"\nPor prioridad:")
for p in ['Alta', 'Media', 'Baja', 'No contactar']:
    n = (df_latest['prioridad'] == p).sum()
    print(f"  {p}: {n:,} ({n/len(df_latest)*100:.0f}%)")

print(f"\nPor objetivo:")
print(df_latest['objetivo'].value_counts().to_string())

print(f"\nPor acción:")
print(df_latest['accion'].value_counts().to_string())

print(f"\nPor timing:")
print(df_latest['timing'].value_counts().to_string())

print(f"\nPor retailer:")
print(df_latest['retailer_recomendado'].value_counts().to_string())

## 3. Visualización

In [ ]:
# ── Visualización del Decision Engine ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1) Prioridad
df_latest['prioridad'].value_counts().reindex(['Alta','Media','Baja','No contactar']).plot(
    kind='bar', ax=axes[0,0], color=['#e74c3c','#f39c12','#95a5a6','#7f8c8d'])
axes[0,0].set_title('Distribución de Prioridad')
axes[0,0].set_ylabel('N clientes')

# 2) Objetivo
df_latest['objetivo'].value_counts().plot(kind='barh', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Objetivo por Funnel')

# 3) Timing
df_latest['timing'].value_counts().plot(kind='barh', ax=axes[0,2], color='coral')
axes[0,2].set_title('Timing')

# 4) Cluster × Prioridad (heatmap)
ct_cp = pd.crosstab(df_latest['cluster_name'], df_latest['prioridad'], normalize='index') * 100
ct_cp = ct_cp.reindex(columns=['Alta','Media','Baja','No contactar'])
sns.heatmap(ct_cp, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[1,0])
axes[1,0].set_title('% Prioridad por Cluster')

# 5) Funnel × Acción
ct_fa = pd.crosstab(df_latest['funnel_state_at_t0'], df_latest['accion'])
ct_fa.plot(kind='barh', stacked=True, ax=axes[1,1], colormap='Set2')
axes[1,1].set_title('Acción por Funnel')
axes[1,1].legend(fontsize=7, loc='lower right')

# 6) EV por cluster
df_latest.boxplot(column='expected_value', by='cluster_name', ax=axes[1,2])
axes[1,2].set_title('Expected Value por Cluster')
axes[1,2].set_xlabel('')
plt.suptitle('')  # remove auto suptitle

plt.tight_layout()
plt.show()

In [ ]:
# ── Matriz de decisión: Funnel × Cluster ─────────────────────────────────
# Muestra qué acción + timing predomina en cada combinación
decision_matrix = df_latest.groupby(['funnel_state_at_t0', 'cluster_name']).agg(
    n=('cust_id', 'count'),
    pct_alta=('prioridad', lambda x: (x == 'Alta').mean() * 100),
    ev_mean=('expected_value', 'mean'),
    p_canje=('propensity_score', 'mean'),
    accion_top=('accion', lambda x: x.mode().iloc[0] if len(x) > 0 else '-'),
    timing_top=('timing', lambda x: x.mode().iloc[0] if len(x) > 0 else '-'),
).reset_index()

print("MATRIZ DE DECISION (Funnel × Cluster)")
print("="*130)
print(f"{'Funnel':<20} {'Cluster':<22} {'N':>5} {'%Alta':>6} {'EV':>10} {'P(canje)':>9} {'Acción':>35} {'Timing':>25}")
print("-"*130)
for _, r in decision_matrix.sort_values(['funnel_state_at_t0', 'cluster_name']).iterrows():
    accion_short = r['accion_top'][:33] if len(r['accion_top']) > 33 else r['accion_top']
    timing_short = r['timing_top'][:23] if len(r['timing_top']) > 23 else r['timing_top']
    print(f"{r['funnel_state_at_t0']:<20} {r['cluster_name']:<22} {r['n']:>5} {r['pct_alta']:>5.0f}% ${r['ev_mean']:>8,.0f} {r['p_canje']:>8.1%} {accion_short:>35} {timing_short:>25}")

## 4. Listas de Activación

In [ ]:
# ── Listas priorizadas para activación ───────────────────────────────────
# 4 listas según el spec: por EV, retailer, urgencia, reactivación

# Lista 1: Top Expected Value
lista_ev = df_latest.nlargest(50, 'expected_value')[['cust_id', 'expected_value', 'prioridad', 'objetivo', 'retailer_recomendado', 'canal']]
print("LISTA 1: TOP 50 POR EXPECTED VALUE")
print(f"  EV rango: ${lista_ev['expected_value'].min():,.0f} — ${lista_ev['expected_value'].max():,.0f}")
print(f"  Retailers: {lista_ev['retailer_recomendado'].value_counts().to_dict()}")

# Lista 2: Por retailer (top 20 por retailer)
print(f"\nLISTA 2: TOP 20 POR RETAILER")
for ret in ['STOREA', 'STOREB', 'STOREC', 'STORED', 'STOREE']:
    mask = df_latest['retailer_recomendado'] == ret
    top_ret = df_latest.loc[mask].nlargest(20, 'expected_value')
    print(f"  {ret}: {len(top_ret)} clientes, EV mean=${top_ret['expected_value'].mean():,.0f}")

# Lista 3: Urgentes (puntos por vencer o fuga)
urgentes = df_latest[df_latest['timing'].str.startswith('Urgente')]
print(f"\nLISTA 3: URGENTES ({len(urgentes):,} clientes)")
print(f"  Por tipo: {urgentes['timing'].value_counts().to_dict()}")
print(f"  EV mean=${urgentes['expected_value'].mean():,.0f}")

# Lista 4: Reactivación (Dormidos + Fuga)
reactiva = df_latest[
    (df_latest['cluster_name'] == 'Dormidos') |
    (df_latest['funnel_state_at_t0'] == 'FUGA')
]
print(f"\nLISTA 4: REACTIVACION ({len(reactiva):,} clientes)")
print(f"  Dormidos: {(reactiva['cluster_name']=='Dormidos').sum():,}")
print(f"  Fuga: {(reactiva['funnel_state_at_t0']=='FUGA').sum():,}")
print(f"  Overlap: {((reactiva['cluster_name']=='Dormidos') & (reactiva['funnel_state_at_t0']=='FUGA')).sum():,}")

## 5. Resumen Ejecutivo

In [ ]:
# ── Resumen ejecutivo ────────────────────────────────────────────────────
# Use df filtered to latest_t0 for columns not in df_output
df_full_latest = df[df['t0'] == latest_t0]

print("="*80)
print("RESUMEN EJECUTIVO — DECISION ENGINE")
print(f"Snapshot: t0={latest_t0.strftime('%Y-%m-%d')}, {len(df_latest):,} clientes")
print("="*80)

print(f"\n1. PRIORIZACION")
for p in ['Alta', 'Media', 'Baja', 'No contactar']:
    n = (df_latest['prioridad'] == p).sum()
    ev = df_latest.loc[df_latest['prioridad']==p, 'expected_value'].mean()
    mask_p = df_full_latest['prioridad'] == p
    tr = df_full_latest.loc[mask_p, 'treatment'].mean()
    print(f"  {p}: {n:,} clientes | EV=${ev:,.0f} | tasa canje historica={tr:.1%}")

print(f"\n2. SEGMENTOS CONDUCTUALES")
for name in sorted(CLUSTER_NAMES.values()):
    mask = df_latest['cluster_name'] == name
    n = mask.sum()
    if n > 0:
        pct_alta = (df_latest.loc[mask, 'prioridad'] == 'Alta').mean() * 100
        print(f"  {name}: {n:,} | {pct_alta:.0f}% prioridad alta | {ACCION_CLUSTER.get(name, {}).get('accion', '-')}")

print(f"\n3. ACCIONES REQUERIDAS")
print(f"  Urgentes (puntos/fuga): {len(urgentes):,}")
print(f"  Reactivación: {len(reactiva):,}")
print(f"  Alta prioridad + inmediato: {((df_latest['prioridad']=='Alta') & (df_latest['timing'].str.contains('Inmediato'))).sum():,}")

print(f"\n4. INCREMENTALIDAD")
print(f"  Lift GitLab (metodo actual): {df_latest['lift_gitlab_quintil'].mean():.1%} promedio")
print(f"  Uplift X-Learner: ${df_latest['uplift_x'].mean():,.0f} promedio")
print(f"  Expected Value: ${df_latest['expected_value'].mean():,.0f} promedio")

print(f"\n5. CANALES")
for canal in df_latest['canal'].value_counts().index:
    n = (df_latest['canal'] == canal).sum()
    print(f"  {canal}: {n:,} ({n/len(df_latest)*100:.0f}%)")

print(f"\nColumnas de salida: {len(OUTPUT_COLS)}")
print(f"Nota: Con datos mock (1000 clientes), las distribuciones son orientativas.")

---
## Next Steps

- **Fase 3**: Dashboard Streamlit (6 vistas)
- **Producción**: Batch scoring mensual (BQ → Colab → BQ)
- **Feedback loop**: Registrar acciones ejecutadas + resultado
- **Monitoreo**: Data drift, model performance, calibración